### Middleware

Middleware provides a way to more tightly control what happens inside the agent.Middleware is useful for the following:
    
    * Tracking agent behaviour with logging analystics and debugging.
    * Transforming prompts tool selection, and output formatting
    * Adding retries, fallbacks and early termination logic
    * Applying rate limits,quadrails and PII detection

In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

True

In [23]:
os.environ["OPENROUTER_API_KEY"] = os.getenv("OPENROUTER_API_KEY")
from langchain.chat_models import init_chat_model
model = init_chat_model(
    "ollama:gemma3:1b",
)
model

ChatOllama(metadata={'lc_versions': {'langchain-core': '1.6.0', 'langchain': '1.3.17'}}, model='gemma3:1b')

### Summarization

Automatically summarize conversation history when approaching token limits, preserving recent messages while compressing older context.Summarization is useful for the following:
* Long running conversations that exceeded context windows.
* Multi turn dialogues with extensive history.
* Applications where preserving full conversation context matters.

In [24]:
from langchain.agents import create_agent
from langchain.agents.middleware import  SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage,SystemMessage


create_agent=create_agent(model=model,checkpointer=InMemorySaver(),middleware=[
    SummarizationMiddleware(model=model,
                            trigger=("messages",10),
                            keep=("messages",4))]
                            )

In [15]:
### Run with thread id

config={"configurable":{"thread_id":"test-1"}}

In [26]:
questions=[
    "What is 2+2?",
    "what is 10*5??",
    "what is 100/4?",
    "What is 15-7?",
    "what is 3*3?",
    "what is 4*4?"
    ]

for q in questions:
    response=create_agent.invoke({"messages":[HumanMessage(content=q)]},config)
    print(f"Messages: {response}")
    print(f"Messages: {len(response['messages'])}")

Messages: {'messages': [HumanMessage(content='What is 2+2?', additional_kwargs={}, response_metadata={}, id='f9bcac40-36d4-486d-98e8-4e6d7b2693ac'), AIMessage(content='2 + 2 = 4 \n\nIt’s a basic arithmetic fact! 😊\n\nWould you like to try another math problem?\n', additional_kwargs={}, response_metadata={'model': 'gemma3:1b', 'created_at': '2026-09-07T19:38:34.5496406Z', 'done': True, 'done_reason': 'stop', 'total_duration': 27585922300, 'load_duration': 14747284600, 'prompt_eval_count': 17, 'prompt_eval_duration': 10239713000, 'eval_count': 30, 'eval_duration': 2573609000, 'logprobs': None, 'model_name': 'gemma3:1b', 'model_provider': 'ollama'}, id='lc_run--01a07d60-bcf1-7593-904a-db10769f12c2-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 17, 'output_tokens': 30, 'total_tokens': 47}), HumanMessage(content='What is 2+2?', additional_kwargs={}, response_metadata={}, id='2b234c46-cce9-4a03-9a43-5b26d9e79e99'), AIMessage(content='4\n', additional_kwargs={}, res

In [27]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain_core import tools
from langchain_core.messages import HumanMessage
from langgraph.checkpoint.memory import InMemorySaver



@tools
def search_hotel(city:str)-> str:
    """Search hotels - returns long response to use more tokens."""
    return f"""Hotels in {city}:
    1: Grand Hotel -5 star, $350/night,spa,pool, gym
    2: City Inn - 4 star,$180/night,business centre
    3: Budget stay - 3 star, $75/night,free wifi
    """

create_agent = create_agent(model="ollama:gemma3:1b",
                            tools=[search_hotel],checkpointer=InMemorySaver(),
                            middleware=[
                                SummarizationMiddleware(
                                    model="ollama:gemma3:1b",
                                    trigger=("tokens",550),
                                    keep=("token",200),
                                    )
                                    ]
                                    )


config= {"configurable": {"thread_id":"test-1"}}

def count_tokens(messges):
    total_chars = sum(len(str(m.content)) for m in messges)
    return total_chars //4

TypeError: 'module' object is not callable